In [1]:
import os
import html
import time

import numpy as np
import pandas as pd
from scipy import sparse

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

# ---- inputs (path strings) ----
RECIPES_PKL = "recipes_labeled.pkl"
TRAIN_PKL = "interactions_train.pkl"
STATS_CSV = "recipe_stats_train.csv"
CATALOG_CSV = "catalog_ids.csv"
EVAL_TEST_PKL = "eval_test_catalog.pkl"
EVAL_USERS_CSV = "eval_users_sample.csv"
RESULTS_07_CSV = "results_07.csv"

# ---- outputs (path strings) ----
RESULTS_08_CSV = "results_08.csv"
RESULTS_ALL_CSV = "results_07_08.csv"
ITEMKNN_NPZ = "itemknn_best.npz"
ITEMKNN_IDS_CSV = "itemknn_ids.csv"

# ---- decisions ----
K = 10                         # top 10
K_NEIGHBORS = 50               # default number of neighbours
K_GRID = [10, 20, 50, 100]     # k values tried for the best variant
K_MAX = max(K_GRID)
BATCH = 250                    # users per batch (memory)
CHUNK = 500                    # catalog recipes per chunk (Item-Based)

for p in [RECIPES_PKL, TRAIN_PKL, STATS_CSV, CATALOG_CSV, EVAL_TEST_PKL, EVAL_USERS_CSV, RESULTS_07_CSV]:
    assert isinstance(p, str), f"{p!r} must be a path string"
    assert os.path.exists(p), f"{p} not found: run notebook 07 first, or fix the path"

In [2]:
recipes = pd.read_pickle(RECIPES_PKL).reset_index(drop=True)
recipes["RecipeId"] = recipes["RecipeId"].astype("int64")
recipes["Name"] = recipes["Name"].astype(str).map(html.unescape)

train = pd.read_pickle(TRAIN_PKL)
train["user_id"] = train["user_id"].astype("int64")
train["recipe_id"] = train["recipe_id"].astype("int64")
stats = pd.read_csv(STATS_CSV)
stats["recipe_id"] = stats["recipe_id"].astype("int64")

catalog_ids = pd.read_csv(CATALOG_CSV)["recipe_id"].astype("int64").to_numpy()
test_cat = pd.read_pickle(EVAL_TEST_PKL)
eval_users = pd.read_csv(EVAL_USERS_CSV)["user_id"].astype("int64").to_numpy()
results_07 = pd.read_csv(RESULTS_07_CSV)

n_cat = len(catalog_ids)
cat_set = set(catalog_ids)
cat_index = pd.Series(np.arange(n_cat), index=catalog_ids)
n_eval = len(eval_users)
u_pos = pd.Series(np.arange(n_eval), index=eval_users)
sample_label = "all" if n_eval == test_cat["user_id"].nunique() else f"sample {n_eval}"

print(f"recipes: {len(recipes):,} | train: {len(train):,} | catalog: {n_cat:,} | eval users: {n_eval:,} ({sample_label})")

recipes: 42,736 | train: 146,775 | catalog: 12,375 | eval users: 2,392 (all)


In [3]:
cat_col = "RecipeCategory"
cat_series = recipes.set_index("RecipeId")[cat_col].fillna("(missing)")
liked_ids = train.loc[train["liked"], "recipe_id"]
liked_ids = liked_ids[liked_ids.isin(cat_series.index)]

dist = pd.DataFrame({
    "all_%": cat_series.value_counts(normalize=True) * 100,
    "catalog_%": cat_series.loc[catalog_ids].value_counts(normalize=True) * 100,
    "liked_train_%": cat_series.loc[liked_ids].value_counts(normalize=True) * 100,
}).fillna(0).round(1).sort_values("catalog_%", ascending=False)
dist["catalog_vs_all"] = (dist["catalog_%"] / dist["all_%"].replace(0, np.nan)).round(2)
print(f"number of categories: {len(dist)}")
dist.head(25)

number of categories: 231


,all_%,catalog_%,liked_train_%,catalog_vs_all
RecipeCategory,,,,
Dessert,19.0,23.1,16.5,1.22
Breads,5.8,7.1,5.2,1.22
Quick Breads,4.8,6.9,9.4,1.44
Breakfast,5.3,4.7,5.4,0.89
Vegetable,5.5,4.4,5.6,0.80
Lunch/Snacks,3.3,3.3,3.5,1.00
Bar Cookie,2.2,2.8,2.2,1.27
Potato,2.1,2.7,2.4,1.29
One Dish Meal,2.2,2.7,2.3,1.23


In [4]:
# check the implausible column (old note said 3,552; notebook 07 found 10,145)
imp = recipes["implausible"].fillna(False).astype(bool)
print(f"implausible = True: {imp.sum():,} of {len(recipes):,} ({imp.mean()*100:.1f}%)")
print(pd.crosstab(recipes["quality"], imp, normalize="index").mul(100).round(1).rename(columns={True: "implausible_%", False: "ok_%"}))

implausible = True: 10,145 of 42,736 (23.7%)
implausible  ok_%  implausible_%
quality                         
fair         84.2           15.8
good         86.3           13.7
poor         76.2           23.8
unverified   68.7           31.3


In [5]:
tc = test_cat[test_cat["user_id"].isin(u_pos.index)]
relevant = (tc.assign(ci=cat_index.loc[tc["recipe_id"]].to_numpy())
              .groupby("user_id")["ci"].apply(np.array).to_dict())
tr = train[train["user_id"].isin(u_pos.index) & train["recipe_id"].isin(cat_set)]
seen = (tr.assign(ci=cat_index.loc[tr["recipe_id"]].to_numpy())
          .groupby("user_id")["ci"].apply(np.array).to_dict())

_disc = 1.0 / np.log2(np.arange(2, K + 2))
_train_ids = set(train["recipe_id"])
cold_ci = cat_index.loc[[r for r in catalog_ids if r not in _train_ids]].to_numpy()


def recommend(score_fn):
    out = np.empty((n_eval, K), dtype=np.int64)
    for start in range(0, n_eval, BATCH):
        rows = np.arange(start, min(start + BATCH, n_eval))
        S = np.array(score_fn(rows), dtype=np.float32, copy=True)
        for r_, row in enumerate(rows):
            s_ = seen.get(eval_users[row])
            if s_ is not None:
                S[r_, s_] = -np.inf
        top = np.argpartition(-S, K, axis=1)[:, :K]
        order_ = np.argsort(-np.take_along_axis(S, top, axis=1), axis=1)
        out[rows] = np.take_along_axis(top, order_, axis=1)
    return out


def evaluate(recs, name):
    hits = np.zeros(n_eval, dtype=bool)
    ndcg = np.zeros(n_eval)
    cold_hit_count = 0
    for row, u in enumerate(eval_users):
        rel = relevant[u]
        h = np.isin(recs[row], rel)
        hits[row] = h.any()
        cold_hit_count += int(np.isin(recs[row][h], cold_ci).sum())
        ndcg[row] = (h * _disc).sum() / _disc[:min(len(rel), K)].sum()
    return {"system": name,
            f"HitRate@{K}": round(float(hits.mean()), 4),
            f"NDCG@{K}": round(float(ndcg.mean()), 4),
            "Coverage_%": round(len(np.unique(recs)) / n_cat * 100, 2),
            "hit_users": int(hits.sum()),
            "cold_hits": cold_hit_count,
            "eval_users": sample_label}


def most_repeated(recs, n=5):
    ids, cnt = np.unique(recs, return_counts=True)
    top = np.argsort(-cnt)[:n]
    rid = catalog_ids[ids[top]]
    return pd.DataFrame({"Name": recipes.set_index("RecipeId").loc[rid, "Name"].to_numpy(),
                         "Category": cat_series.loc[rid].to_numpy(),
                         "users_%": (cnt[top] / n_eval * 100).round(1)})


# Popularity (same as notebook 07): a framework check, and a fill-in when there are too few neighbours
st = stats.set_index("recipe_id").reindex(catalog_ids)
wr = st["weighted_rating"].fillna(-1).to_numpy()
ni = st["n_interactions"].fillna(0).to_numpy()
pop_order = np.lexsort((ni, wr))
pop_score = np.empty(n_cat, dtype=np.float32)
pop_score[pop_order] = np.arange(n_cat)
pop_tiebreak = (pop_score / n_cat * 1e-6).astype(np.float32)   # tiny value: only breaks ties between recipes with equal (or zero) CF score

recs_pop = recommend(lambda rows: np.broadcast_to(pop_score, (len(rows), n_cat)))
check = evaluate(recs_pop, "Popularity")
ref = results_07.set_index("system").loc["Popularity"]
print("Popularity here:       ", check[f"HitRate@{K}"], check[f"NDCG@{K}"], check["hit_users"])
print("Popularity notebook 07:", ref[f"HitRate@{K}"], ref[f"NDCG@{K}"], ref["hit_users"])
assert check["hit_users"] == ref["hit_users"], "framework does not match notebook 07!"
print("✓ framework matches notebook 07")

Popularity here:        0.0134 0.0043 32
Popularity notebook 07: 0.0134 0.0043 32
✓ framework matches notebook 07


In [6]:
all_rids = recipes["RecipeId"].to_numpy()
rid_to_col = pd.Series(np.arange(len(all_rids)), index=all_rids)
n_items = len(all_rids)

tr_all = train[train["recipe_id"].isin(rid_to_col.index)].copy()
train_users = np.sort(tr_all["user_id"].unique())
uid_to_row = pd.Series(np.arange(len(train_users)), index=train_users)
n_users = len(train_users)
tr_all["r"] = uid_to_row.loc[tr_all["user_id"]].to_numpy()
tr_all["c"] = rid_to_col.loc[tr_all["recipe_id"]].to_numpy()


def mat(df_, vals):
    return sparse.csr_matrix((vals, (df_["r"].to_numpy(), df_["c"].to_numpy())), shape=(n_users, n_items), dtype=np.float32)


B = mat(tr_all, np.ones(len(tr_all)))
lk = tr_all[tr_all["liked"]]
L_all = mat(lk, np.ones(len(lk)))

rt = tr_all[tr_all["has_rating"]].copy()
item_mean = rt.groupby("c")["rating"].transform("mean")
Rc = mat(rt, (rt["rating"] - item_mean).to_numpy())
Rc.eliminate_zeros()

eval_rows_in_train = uid_to_row.loc[eval_users].to_numpy()   # every test user has training rows (checked in notebook 06)
L_eval = L_all[eval_rows_in_train]
no_profile = np.asarray(L_eval.sum(axis=1)).ravel() == 0

cat_cols = rid_to_col.loc[catalog_ids].to_numpy()
print(f"B: {n_users:,} users × {n_items:,} recipes | nnz {B.nnz:,}")
print(f"Rc (stars, centered): nnz {Rc.nnz:,} ({Rc.nnz / max(len(rt),1) * 100:.1f}% of star ratings are non-zero after centring)")
print(f"eval users with no liked training recipe (fall back to Popularity): {no_profile.sum():,} ({no_profile.mean()*100:.1f}%)")

B: 60,183 users × 42,736 recipes | nnz 146,775
Rc (stars, centered): nnz 109,058 (79.4% of star ratings are non-zero after centring)
eval users with no liked training recipe (fall back to Popularity): 2 (0.1%)


In [7]:
def item_neighbors(M, kind):
    """Top K_MAX neighbours (from all recipes) for each catalog recipe. Returns (indices, values), shape (n_cat × K_MAX), strongest first."""
    Mt = M.T.tocsr()                                         # recipes × users
    if kind == "jaccard":
        cnt = np.asarray(M.sum(axis=0)).ravel()              # how many users interacted with each recipe
    else:
        norm = np.sqrt(np.asarray(M.multiply(M).sum(axis=0)).ravel())
    idx = np.zeros((n_cat, K_MAX), dtype=np.int64)
    val = np.zeros((n_cat, K_MAX), dtype=np.float32)
    t0 = time.time()
    for s in range(0, n_cat, CHUNK):
        cols = cat_cols[s:s + CHUNK]
        co = (Mt @ M[:, cols]).toarray()                     # recipes × chunk
        if kind == "jaccard":
            denom = cnt[:, None] + cnt[cols][None, :] - co
            sim = np.divide(co, denom, out=np.zeros_like(co), where=denom > 0)
        else:
            denom = norm[:, None] * norm[cols][None, :]
            sim = np.divide(co, denom, out=np.zeros_like(co), where=denom > 0)
        sim[cols, np.arange(len(cols))] = 0                  # a recipe is not its own neighbour
        sim[sim < 0] = 0                                     # keep positive similarity only
        top = np.argpartition(-sim, K_MAX, axis=0)[:K_MAX]   # K_MAX × chunk
        tv = np.take_along_axis(sim, top, axis=0)
        o = np.argsort(-tv, axis=0)
        idx[s:s + CHUNK] = np.take_along_axis(top, o, axis=0).T
        val[s:s + CHUNK] = np.take_along_axis(tv, o, axis=0).T
    print(f"  {kind}: {time.time() - t0:.0f}s | catalog recipes with no neighbour: {(val[:, 0] == 0).sum():,}")
    return idx, val


def knn_matrix(idx, val, k):
    """Sparse matrix (all recipes × catalog) keeping only the top-k neighbours."""
    rows = idx[:, :k].ravel()
    cols = np.repeat(np.arange(n_cat), k)
    v = val[:, :k].ravel()
    keep = v > 0
    return sparse.csr_matrix((v[keep], (rows[keep], cols[keep])), shape=(n_items, n_cat))


def item_scores_fn(S_k):
    def fn(rows):
        S = (L_eval[rows] @ S_k).toarray()
        return S + pop_tiebreak
    return fn


item_nb = {}
print("computing neighbours:")
item_nb["Item-Based Cosine (implicit)"] = item_neighbors(B, "cosine")
item_nb["Item-Based Jaccard (implicit)"] = item_neighbors(B, "jaccard")
item_nb["Item-Based Pearson (stars)"] = item_neighbors(Rc, "cosine")

computing neighbours:
  cosine: 4s | catalog recipes with no neighbour: 6,428
  jaccard: 4s | catalog recipes with no neighbour: 6,428
  cosine: 3s | catalog recipes with no neighbour: 9,945


In [8]:
results = []
recs_store = {}
for name, (idx, val) in item_nb.items():
    recs_ = recommend(item_scores_fn(knn_matrix(idx, val, K_NEIGHBORS)))
    recs_store[name] = recs_
    results.append(evaluate(recs_, f"{name}, k={K_NEIGHBORS}"))
pd.DataFrame(results).set_index("system")

,HitRate@10,NDCG@10,Coverage_%,hit_users,cold_hits,eval_users
system,,,,,,
"Item-Based Cosine (implicit), k=50",0.0059,0.0019,28.67,14,0,all
"Item-Based Jaccard (implicit), k=50",0.0234,0.0096,34.72,56,0,all
"Item-Based Pearson (stars), k=50",0.0092,0.0020,15.43,22,0,all


In [9]:
B_cnt = np.asarray(B.sum(axis=1)).ravel()
B_norm = np.sqrt(B_cnt)
B_T = B.T.tocsr()
L_cat = L_all[:, cat_cols].tocsr()                          # what each user liked from the catalog


def user_sims(rows, kind):
    co = (L_eval[rows] @ B_T).toarray()                      # eval batch × all train users
    p_cnt = np.asarray(L_eval[rows].sum(axis=1)).ravel()
    if kind == "jaccard":
        denom = p_cnt[:, None] + B_cnt[None, :] - co
    else:
        denom = np.sqrt(p_cnt)[:, None] * B_norm[None, :]
    sim = np.divide(co, denom, out=np.zeros_like(co), where=denom > 0)
    sim[np.arange(len(rows)), eval_rows_in_train[rows]] = 0  # a user is not her own neighbour
    return sim


def user_scores_fn(kind, k):
    def fn(rows):
        sim = user_sims(rows, kind)
        top = np.argpartition(-sim, k, axis=1)[:, :k]
        W = np.zeros_like(sim)
        np.put_along_axis(W, top, np.take_along_axis(sim, top, axis=1), axis=1)
        S = sparse.csr_matrix(W) @ L_cat
        return np.asarray(S.todense()) + pop_tiebreak
    return fn


for name, kind in [("User-Based Cosine (implicit)", "cosine"), ("User-Based Jaccard (implicit)", "jaccard")]:
    t0 = time.time()
    recs_ = recommend(user_scores_fn(kind, K_NEIGHBORS))
    recs_store[name] = recs_
    results.append(evaluate(recs_, f"{name}, k={K_NEIGHBORS}"))
    print(f"{name}: {time.time() - t0:.0f}s")
pd.DataFrame(results).set_index("system")

User-Based Cosine (implicit): 1s
User-Based Jaccard (implicit): 1s


,HitRate@10,NDCG@10,Coverage_%,hit_users,cold_hits,eval_users
system,,,,,,
"Item-Based Cosine (implicit), k=50",0.0059,0.0019,28.67,14,0,all
"Item-Based Jaccard (implicit), k=50",0.0234,0.0096,34.72,56,0,all
"Item-Based Pearson (stars), k=50",0.0092,0.0020,15.43,22,0,all
"User-Based Cosine (implicit), k=50",0.0326,0.0104,11.27,78,0,all
"User-Based Jaccard (implicit), k=50",0.0790,0.0288,15.94,189,0,all


In [10]:
res_k50 = pd.DataFrame(results)
res_k50["base"] = res_k50["system"].str.replace(f", k={K_NEIGHBORS}", "", regex=False)


def best_of(prefix):
    sub = res_k50[res_k50["base"].str.startswith(prefix)]
    return sub.sort_values([f"NDCG@{K}", f"HitRate@{K}"], ascending=False).iloc[0]["base"]


best_item = best_of("Item-Based")
best_user = best_of("User-Based")
print("best Item-Based:", best_item)
print("best User-Based:", best_user)

k_rows = []
for k in K_GRID:
    idx, val = item_nb[best_item]
    k_rows.append(evaluate(recommend(item_scores_fn(knn_matrix(idx, val, k))), f"{best_item}, k={k}"))
    kind = "jaccard" if "Jaccard" in best_user else "cosine"
    k_rows.append(evaluate(recommend(user_scores_fn(kind, k)), f"{best_user}, k={k}"))
res_grid = pd.DataFrame(k_rows).set_index("system")
res_grid

best Item-Based: Item-Based Jaccard (implicit)
best User-Based: User-Based Jaccard (implicit)


,HitRate@10,NDCG@10,Coverage_%,hit_users,cold_hits,eval_users
system,,,,,,
"Item-Based Jaccard (implicit), k=10",0.0468,0.0177,28.29,112,0,all
"User-Based Jaccard (implicit), k=10",0.0573,0.0186,14.63,137,0,all
"Item-Based Jaccard (implicit), k=20",0.0314,0.0118,32.10,75,0,all
"User-Based Jaccard (implicit), k=20",0.0627,0.0223,14.80,150,0,all
"Item-Based Jaccard (implicit), k=50",0.0234,0.0096,34.72,56,0,all
"User-Based Jaccard (implicit), k=50",0.0790,0.0288,15.94,189,0,all
"Item-Based Jaccard (implicit), k=100",0.0184,0.0079,33.85,44,0,all
"User-Based Jaccard (implicit), k=100",0.0920,0.0353,16.27,220,0,all


In [11]:
def best_k(base):
    sub = res_grid[res_grid.index.str.startswith(base + ",")]
    best = sub.sort_values([f"NDCG@{K}", f"HitRate@{K}"], ascending=False).index[0]
    return int(best.split("k=")[1])


k_item = best_k(best_item)
k_user = best_k(best_user)
print(f"chosen k: {best_item} → {k_item} | {best_user} → {k_user}")

final_item_recs = recommend(item_scores_fn(knn_matrix(*item_nb[best_item], k_item)))
kind_u = "jaccard" if "Jaccard" in best_user else "cosine"
final_user_recs = recommend(user_scores_fn(kind_u, k_user))

chosen k: Item-Based Jaccard (implicit) → 10 | User-Based Jaccard (implicit) → 100


In [12]:
n_liked_eval = np.asarray(L_eval.sum(axis=1)).ravel()
ex_row = int(np.argmax(np.where(n_liked_eval <= 15, n_liked_eval, 0)))
ex_user = eval_users[ex_row]
liked_ex = lk.loc[lk["user_id"] == ex_user, "recipe_id"]
print(f"user {ex_user}: {len(liked_ex)} liked recipes in training")
print(recipes.set_index("RecipeId").loc[liked_ex, ["Name", cat_col]].head(10).to_string())


def show(recs, row):
    ids = catalog_ids[recs[row]]
    return pd.DataFrame({"Name": recipes.set_index("RecipeId").loc[ids, "Name"].to_numpy(),
                         "Category": cat_series.loc[ids].to_numpy(),
                         "hidden_hit": np.isin(recs[row], relevant[ex_user])})


print(f"\n{best_item} (k={k_item}):")
print(show(final_item_recs, ex_row).to_string())
print(f"\n{best_user} (k={k_user}):")
print(show(final_user_recs, ex_row).to_string())

user 12882: 15 liked recipes in training
                                                       Name   RecipeCategory
RecipeId                                                                    
3094                           Confectioner's Sugar Cookies          Dessert
8953                                Cream Cheese Pound Cake          Dessert
9973                       Pineapple and Sour Cream Muffins     Quick Breads
12736                                 Orange Glazed Chicken          Chicken
14059                           Lennie's Special Shortbread          Dessert
14467                                         Pecan Tassies              Pie
19542                                  Blue Cheese Dressing  Salad Dressings
21812                                   Peanut Butter Balls            Candy
24521                                         Blueberry Pie              Pie
26306     Copycat  Green Giant Niblets Corn in Butter Sauce             Corn

Item-Based Jaccard (implicit) (k=1

In [13]:
res_08 = pd.concat([pd.DataFrame(results).set_index("system"), res_grid])
res_08 = res_08[~res_08.index.duplicated()]

final_rows = [evaluate(final_item_recs, f"{best_item}, k={k_item} (best)"),
              evaluate(final_user_recs, f"{best_user}, k={k_user} (best)")]
summary = pd.concat([results_07.set_index("system"), pd.DataFrame(final_rows).set_index("system")])
summary

,HitRate@10,NDCG@10,Coverage_%,hit_users,cold_hits,eval_users
system,,,,,,
Random,0.0021,0.0005,85.60,5,0,all
Popularity,0.0134,0.0043,0.11,32,0,all
Content-Based A: mean profile,0.0038,0.0009,31.68,9,1,all
Content-Based B: max-similarity,0.0063,0.0016,32.90,15,0,all
"Item-Based Jaccard (implicit), k=10 (best)",0.0468,0.0177,28.29,112,0,all
"User-Based Jaccard (implicit), k=100 (best)",0.0920,0.0353,16.27,220,0,all


In [14]:
for nm, rc in [(f"{best_item}, k={k_item}", final_item_recs), (f"{best_user}, k={k_user}", final_user_recs)]:
    print(f"\n{nm}: top 5 most repeated recipes")
    print(most_repeated(rc).to_string(index=False))


Item-Based Jaccard (implicit), k=10: top 5 most repeated recipes
                                                 Name     Category  users_%
                                    Cream Cheese Flan      Dessert     13.8
                                  Banana Banana Bread Quick Breads     13.6
                                       Snickerdoodles      Dessert     12.6
Absolutely the Best New York Cheesecake (Gluten-Free)   Cheesecake     10.8
                       Italian Style Skillet Zucchini    Vegetable      9.3

User-Based Jaccard (implicit), k=100: top 5 most repeated recipes
                                                 Name     Category  users_%
                               Oatmeal Raisin Cookies Drop Cookies     41.2
                                    Cream Cheese Flan      Dessert     37.5
                                       Snickerdoodles      Dessert     36.6
Absolutely the Best New York Cheesecake (Gluten-Free)   Cheesecake     35.8
                               

In [15]:
res_08.reset_index().to_csv(RESULTS_08_CSV, index=False, encoding="utf-8-sig")
summary.reset_index().to_csv(RESULTS_ALL_CSV, index=False, encoding="utf-8-sig")

# neighbour matrix of the best Item-Based (at K_MAX; the Hybrid trims to the chosen k)
idx, val = item_nb[best_item]
sparse.save_npz(ITEMKNN_NPZ, knn_matrix(idx, val, K_MAX))
pd.DataFrame({"recipe_id": all_rids}).to_csv(ITEMKNN_IDS_CSV, index=False, encoding="utf-8-sig")
pd.Series({"best_item": best_item, "k_item": k_item, "best_user": best_user, "k_user": k_user}).to_csv("cf_choice.csv", encoding="utf-8-sig")
print("saved:", RESULTS_08_CSV, RESULTS_ALL_CSV, ITEMKNN_NPZ, ITEMKNN_IDS_CSV, "cf_choice.csv")

saved: results_08.csv results_07_08.csv itemknn_best.npz itemknn_ids.csv cf_choice.csv
